# TSFabrics pilot: Stage 1 (CPU, no images)

This notebook only **launches** `scripts/run_stage1.py`. All logic lives in the repository.

**Before running**
1. Attach the TSFabrics dataset (the same one used for the forensic audit).
2. Make the repository available: either attach it as a Kaggle dataset named `tsfabrics-alignment-pilot` (upload the zip), or set `GIT_URL` below to your GitHub repo.
3. Settings: **CPU** is enough. **Internet ON** (for the pinned packages).

**What Stage 1 reads:** synthetic data, the frozen CSVs and the per-scenario label CSVs. It opens **no image** and computes no detector output.

**What to send back:** the printed summary at the end, plus `audit/stage1_report.json` and `audit/bank_frames_v1_2_1.csv` from the Output panel.

In [ ]:
import os, shutil, subprocess, sys, glob, json
GIT_URL = ""   # e.g. "https://github.com/Tinonturja/tsfabrics-alignment-pilot.git" once the repo exists
WORK = "/kaggle/working/repo"
if os.path.exists(WORK):
    shutil.rmtree(WORK)
if GIT_URL:
    subprocess.run(["git", "clone", "--depth", "1", GIT_URL, WORK], check=True)
else:
    cands = [p for p in glob.glob("/kaggle/input/**/PREREGISTRATION.md", recursive=True)]
    assert cands, "Attach the repository as a dataset, or set GIT_URL"
    shutil.copytree(os.path.dirname(cands[0]), WORK)
print("repo at", WORK, "| files:", len(glob.glob(WORK + "/**/*", recursive=True)))

In [ ]:
# Pinned Stage 1 packages (numpy 2.4.4 is needed for the AT-15 offset hashes). Installed for the subprocess below.
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{WORK}/requirements.txt"],
                   capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])

In [ ]:
# Locate the dataset root (the folder that contains CSV/ and IMAGE/)
root = None
for base, dirs, files in os.walk("/kaggle/input"):
    if "CSV" in dirs and "IMAGE" in dirs:
        root = base; break
print("dataset root:", root)

In [ ]:
# Run Stage 1 in a fresh interpreter (so the pinned packages are the ones imported). About 10 to 20 minutes.
r = subprocess.run([sys.executable, "scripts/run_stage1.py", "--dataset-root", root], cwd=WORK,
                   capture_output=True, text=True)
print(r.stdout[-6000:]); print(r.stderr[-3000:])

In [ ]:
rep = json.load(open(f"{WORK}/audit/stage1_report.json"))
print("RESULT:", rep["result"], rep["counts"])
print("blocking failures:", rep["blocking_failures"])
print("C6 reportable:", rep["c6_reportable"])
print("numpy:", rep["environment"]["numpy"], "| numba:", rep["environment"]["numba"], "| python:", rep["environment"]["python"].split()[0])
for t in rep["tests"]:
    if t["status"] != "passed":
        print(t["status"].upper(), t["module"], t["test"])
os.makedirs("/kaggle/working/stage1_out", exist_ok=True)
for f in ("audit/stage1_report.json", "audit/bank_frames_v1_2_1.csv"):
    if os.path.exists(f"{WORK}/{f}"):
        shutil.copy(f"{WORK}/{f}", "/kaggle/working/stage1_out/")
print(os.listdir("/kaggle/working/stage1_out"))